---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-37: LangChain <b>Retrievers</b></h1>

# Learning agenda of this notebook

> **The idea of this lecture:** a vector store (Lec-36) can find *similar* text, but in Example 6 of Lec-36 it did not find the *right* text for "Who are the authors of the paper?". A **retriever** is any component that takes a question and returns a list of Documents. Some search a vector store, some search the web, some rewrite the question, some re-score the results, and some do not use embeddings at all. In production, several are combined.

1. **What is a Retriever?**
2. **Setup: Two Stores from Lec-36**
3. **Retrievers on a Vector Store**
    - Example 1: `as_retriever()`: similarity search
    - Example 2: Only good matches: a score threshold
    - Example 3: Relevant but different: MMR
4. **Retrievers without a Vector Store**
    - Example 4: `WikipediaRetriever`
5. **Better Questions, Better Ranking**
    - Example 5: `MultiQueryRetriever`: an LLM asks the question in several ways
    - Example 6: Reranking: fetch many, keep the best
6. **Keyword Search and Hybrid Search**
    - Example 7: BM25 by hand, then with `BM25Retriever`
    - Example 8: `BM25Retriever` on a transcript, then BM25 + vectors
7. **Summary**

> **Note:** Example 4 needs an internet connection and Example 5 uses the free `openai/gpt-oss-120b` model on Groq. Everything else runs locally; Example 6 downloads a small reranking model (about 90 MB) the first time.

# <span style='background :lightgreen' >1. What is a Retriever?</span>

In the RAG app of Lec-30, the retriever is the first step for every question: it decides which chunks the model will read.

<div style="text-align:center;">
    <img src="../images/langchain-new-03.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

A retriever has one job: **a question in, a list of `Document`s out**. It is a Runnable (Lec-33), so it has `invoke()` and `batch()` and can be joined with `|`. Unlike a vector store, it has no `add_documents()`: it only finds.

<div style="text-align:center;">
    <img src="../images/langchain-new-28.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

**More retrievers you will meet** (not in this lecture):

| Retriever | Idea | Package |
|---|---|---|
| `ParentDocumentRetriever` | search small chunks, but return the larger chunk (or page) they come from | `langchain-classic` |
| `SelfQueryRetriever` | an LLM turns "papers after 2020 about vision" into a search **plus a metadata filter** | `langchain-classic` |
| `MultiVectorRetriever` | store several vectors per Document (summary, questions, text) | `langchain-classic` |
| `TimeWeightedVectorStoreRetriever` | prefer recently used Documents | `langchain-classic` |
| `TavilySearchAPIRetriever` and other search APIs | search the live web | `langchain-tavily` and others |

> **Note:** in LangChain 1.x a retriever is called with `invoke()`. The old method `get_relevant_documents()` was removed. Many advanced retrievers now live in the `langchain-classic` package.

# <span style='background :lightgreen' >2. Setup: Two Stores from Lec-36</span>

Every example below is **standalone**: its first code cell rebuilds, in memory, the store of Lec-36 it needs (about two seconds), and its later cells build on it, so run them in order:
- `facts`: the 14 short facts (Lec-36, Examples 1 to 5), in Chroma with the cosine distance (Examples 1 to 3);
- `paper`: the 60 chunks of `attention.pdf` (Lec-36, Example 6), in an `InMemoryVectorStore` (Section 5, Examples 5 and 6).

Example 4 searches Wikipedia, and Examples 7 and 8 build their own small collections.

# <span style='background :lightgreen' >3. Retrievers on a Vector Store</span>

## <span style='color:#7e22ce'>Example 1: <code>as_retriever()</code>: similarity search</span>

`as_retriever()` turns any vector store into a retriever. Two arguments decide how it searches:
- `search_type`: `"similarity"` (default), `"similarity_score_threshold"` or `"mmr"`;
- `search_kwargs`: the settings of that search, for example `{"k": 3}`.

In [1]:
# Step a: rebuild the 14 facts of Lec-36 in Chroma, and a helper to print results
import transformers                                         # the library under sentence-transformers
transformers.logging.set_verbosity_error()                  # hide library warnings
transformers.logging.disable_progress_bar()                 # hide the "Loading weights" progress bar
from langchain_huggingface import HuggingFaceEmbeddings     # the free local embedding model
from langchain_core.documents import Document
from langchain_chroma import Chroma                         # the vector store of Lec-36

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

docs = [
    Document("Islamabad is the capital of Pakistan", metadata={"category": "Geography", "country": "Pakistan"}),
    Document("Paris is the capital of France", metadata={"category": "Geography", "country": "France"}),
    Document("Rome is the capital of Italy", metadata={"category": "Geography", "country": "Italy"}),
    Document("LangChain makes it easy to work with LLMs.", metadata={"category": "AI Tools"}),
    Document("LangChain is used to build LLM based applications.", metadata={"category": "AI Tools"}),
    Document("Chroma is used to store and search document embeddings.", metadata={"category": "Vector Database"}),
    Document("Embeddings are vector representations of text.", metadata={"category": "AI Concepts"}),
    Document("MMR helps you get diverse results when doing similarity search.", metadata={"category": "Search"}),
    Document("LangChain supports Chroma, FAISS, Pinecone, and more.", metadata={"category": "AI Tools"}),
    Document("Climate change is causing glaciers to melt rapidly in the Arctic region.", metadata={"category": "Environment", "region": "Arctic"}),
    Document("Glaciers in the Arctic are melting at an alarming rate due to rising temperatures.", metadata={"category": "Environment", "region": "Arctic"}),
    Document("Deforestation in the Amazon is accelerating global climate change.", metadata={"category": "Environment", "region": "Amazon"}),
    Document("Climate change is increasing the frequency of wildfires in California.", metadata={"category": "Environment", "region": "California"}),
    Document("Rising sea levels due to climate change threaten coastal cities like Mumbai and New York.", metadata={"category": "Environment", "region": "Global"}),
]

ids = []
for number in range(1, 15):
    ids.append(f"doc-{number:02d}")                         # "doc-01" ... "doc-14"

facts = Chroma.from_documents(
    docs,                                                   # the 14 facts
    embeddings,                                             # turns each fact into a vector
    ids=ids,                                                # one id per fact: building it again adds no duplicates
    collection_name="facts",                                # the collection's name
    collection_metadata={"hnsw:space": "cosine"},           # compare vectors with the cosine distance
)
print(len(docs), "facts are stored")

def show(results):
    """Print each Document's text, numbered."""
    number = 0
    for doc in results:
        number += 1
        print(f"   {number}. {doc.page_content}")

14 facts are stored


In [2]:
# Step b: as_retriever() turns the store into a retriever
retriever = facts.as_retriever(
    search_type="similarity",                               # the k nearest Documents (the default)
    search_kwargs={"k": 3},                                 # k = 3
)
print(type(retriever).__name__)                             # VectorStoreRetriever

results = retriever.invoke("What is the capital of Pakistan?")   # one question -> a list of Documents
show(results)

VectorStoreRetriever
   1. Islamabad is the capital of Pakistan
   2. Rome is the capital of Italy
   3. Paris is the capital of France


In [3]:
# Step c: a retriever is a Runnable, so batch() works too
questions = ["Which city is the capital of France?", "What are embeddings?"]
all_results = retriever.batch(questions)                    # one list of Documents per question
for results in all_results:
    show(results[:1])                                       # the best Document for each question

   1. Paris is the capital of France
   1. Embeddings are vector representations of text.


The retriever does exactly what `similarity_search()` did in Lec-36, but behind the common retriever interface, so it can be replaced by any other retriever in this lecture without changing the rest of the app.

The answer to "What is the capital of Pakistan?" is the first Document, but `k=3` always returns three Documents, even when the other two (Rome, Paris) are useless.

## <span style='color:#7e22ce'>Example 2: Only good matches: a score threshold</span>

`"similarity_score_threshold"` returns only the Documents whose **relevance score** (Lec-36, Example 2: higher is better, 0 to 1) is above a threshold. It can return fewer than `k` Documents, or none.

In [4]:
# Step a: rebuild the 14 facts of Lec-36 in Chroma, and a helper to print results
import transformers                                         # the library under sentence-transformers
transformers.logging.set_verbosity_error()                  # hide library warnings
transformers.logging.disable_progress_bar()                 # hide the "Loading weights" progress bar
from langchain_huggingface import HuggingFaceEmbeddings     # the free local embedding model
from langchain_core.documents import Document
from langchain_chroma import Chroma                         # the vector store of Lec-36

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

docs = [
    Document("Islamabad is the capital of Pakistan", metadata={"category": "Geography", "country": "Pakistan"}),
    Document("Paris is the capital of France", metadata={"category": "Geography", "country": "France"}),
    Document("Rome is the capital of Italy", metadata={"category": "Geography", "country": "Italy"}),
    Document("LangChain makes it easy to work with LLMs.", metadata={"category": "AI Tools"}),
    Document("LangChain is used to build LLM based applications.", metadata={"category": "AI Tools"}),
    Document("Chroma is used to store and search document embeddings.", metadata={"category": "Vector Database"}),
    Document("Embeddings are vector representations of text.", metadata={"category": "AI Concepts"}),
    Document("MMR helps you get diverse results when doing similarity search.", metadata={"category": "Search"}),
    Document("LangChain supports Chroma, FAISS, Pinecone, and more.", metadata={"category": "AI Tools"}),
    Document("Climate change is causing glaciers to melt rapidly in the Arctic region.", metadata={"category": "Environment", "region": "Arctic"}),
    Document("Glaciers in the Arctic are melting at an alarming rate due to rising temperatures.", metadata={"category": "Environment", "region": "Arctic"}),
    Document("Deforestation in the Amazon is accelerating global climate change.", metadata={"category": "Environment", "region": "Amazon"}),
    Document("Climate change is increasing the frequency of wildfires in California.", metadata={"category": "Environment", "region": "California"}),
    Document("Rising sea levels due to climate change threaten coastal cities like Mumbai and New York.", metadata={"category": "Environment", "region": "Global"}),
]

ids = []
for number in range(1, 15):
    ids.append(f"doc-{number:02d}")                         # "doc-01" ... "doc-14"

facts = Chroma.from_documents(
    docs,                                                   # the 14 facts
    embeddings,                                             # turns each fact into a vector
    ids=ids,                                                # one id per fact: building it again adds no duplicates
    collection_name="facts",                                # the collection's name
    collection_metadata={"hnsw:space": "cosine"},           # compare vectors with the cosine distance
)
print(len(docs), "facts are stored")

def show(results):
    """Print each Document's text, numbered."""
    number = 0
    for doc in results:
        number += 1
        print(f"   {number}. {doc.page_content}")

14 facts are stored


In [5]:
# Step b: keep only the Documents with a relevance score of at least 0.3
strict = facts.as_retriever(
    search_type="similarity_score_threshold",               # filter by relevance score
    search_kwargs={"score_threshold": 0.3, "k": 3},         # keep relevance >= 0.3, at most 3 Documents
)
results = strict.invoke("What is the capital of Pakistan?")
show(results)                                               # Islamabad 0.81, Rome 0.34; Paris 0.21 is dropped

   1. Islamabad is the capital of Pakistan
   2. Rome is the capital of Italy


In [6]:
# Step c: a question that nothing in our facts can answer
nothing = strict.invoke("How do I bake bread?")             # nothing in our facts is about bread
print("bread:", len(nothing), "Documents")                  # an empty list, not three wrong answers

No relevant docs were retrieved using the relevance score threshold 0.3


bread: 0 Documents


LangChain also prints a warning when nothing passes the threshold. A threshold lets the app say *"I could not find this in the documents"* instead of answering from unrelated text. The right value depends on the embedding model and the store's score (Lec-36, figure in Example 2), so test it on your own questions.

## <span style='color:#7e22ce'>Example 3: Relevant but different: MMR</span>

The top `k` results are often **near-copies** of each other: two facts about melting Arctic glaciers say the same thing twice. **Maximal Marginal Relevance (MMR)** picks Documents one at a time: each new one must be relevant to the question **and** different from the ones already picked.
- `fetch_k`: how many candidates to fetch first (by similarity);
- `k`: how many to return from them;
- `lambda_mult`: the balance, from `1.0` (only relevance, like similarity search) to `0.0` (only diversity).

In [7]:
# Step a: rebuild the 14 facts of Lec-36 in Chroma, and a helper to print results
import transformers                                         # the library under sentence-transformers
transformers.logging.set_verbosity_error()                  # hide library warnings
transformers.logging.disable_progress_bar()                 # hide the "Loading weights" progress bar
from langchain_huggingface import HuggingFaceEmbeddings     # the free local embedding model
from langchain_core.documents import Document
from langchain_chroma import Chroma                         # the vector store of Lec-36

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

docs = [
    Document("Islamabad is the capital of Pakistan", metadata={"category": "Geography", "country": "Pakistan"}),
    Document("Paris is the capital of France", metadata={"category": "Geography", "country": "France"}),
    Document("Rome is the capital of Italy", metadata={"category": "Geography", "country": "Italy"}),
    Document("LangChain makes it easy to work with LLMs.", metadata={"category": "AI Tools"}),
    Document("LangChain is used to build LLM based applications.", metadata={"category": "AI Tools"}),
    Document("Chroma is used to store and search document embeddings.", metadata={"category": "Vector Database"}),
    Document("Embeddings are vector representations of text.", metadata={"category": "AI Concepts"}),
    Document("MMR helps you get diverse results when doing similarity search.", metadata={"category": "Search"}),
    Document("LangChain supports Chroma, FAISS, Pinecone, and more.", metadata={"category": "AI Tools"}),
    Document("Climate change is causing glaciers to melt rapidly in the Arctic region.", metadata={"category": "Environment", "region": "Arctic"}),
    Document("Glaciers in the Arctic are melting at an alarming rate due to rising temperatures.", metadata={"category": "Environment", "region": "Arctic"}),
    Document("Deforestation in the Amazon is accelerating global climate change.", metadata={"category": "Environment", "region": "Amazon"}),
    Document("Climate change is increasing the frequency of wildfires in California.", metadata={"category": "Environment", "region": "California"}),
    Document("Rising sea levels due to climate change threaten coastal cities like Mumbai and New York.", metadata={"category": "Environment", "region": "Global"}),
]

ids = []
for number in range(1, 15):
    ids.append(f"doc-{number:02d}")                         # "doc-01" ... "doc-14"

facts = Chroma.from_documents(
    docs,                                                   # the 14 facts
    embeddings,                                             # turns each fact into a vector
    ids=ids,                                                # one id per fact: building it again adds no duplicates
    collection_name="facts",                                # the collection's name
    collection_metadata={"hnsw:space": "cosine"},           # compare vectors with the cosine distance
)
print(len(docs), "facts are stored")

def show(results):
    """Print each Document's text, numbered."""
    number = 0
    for doc in results:
        number += 1
        print(f"   {number}. {doc.page_content}")

14 facts are stored


In [8]:
# Step b: plain similarity search: the 3 most similar facts
question = "Tell me about melting glaciers"

similar = facts.as_retriever(search_kwargs={"k": 3})
print("similarity, k=3:")
show(similar.invoke(question))

similarity, k=3:
   1. Glaciers in the Arctic are melting at an alarming rate due to rising temperatures.
   2. Climate change is causing glaciers to melt rapidly in the Arctic region.
   3. Rising sea levels due to climate change threaten coastal cities like Mumbai and New York.


In [9]:
# Step c: MMR: 3 facts chosen from the 5 most similar
mmr = facts.as_retriever(
    search_type="mmr",                                      # Maximal Marginal Relevance
    search_kwargs={
        "k": 3,                                             # return 3 Documents ...
        "fetch_k": 5,                                       # ... chosen from the 5 most similar
        "lambda_mult": 0.5,                                 # half relevance, half diversity
    },
)
print("MMR, k=3 from the 5 most similar (fetch_k=5):")
show(mmr.invoke(question))

MMR, k=3 from the 5 most similar (fetch_k=5):
   1. Glaciers in the Arctic are melting at an alarming rate due to rising temperatures.
   2. Rising sea levels due to climate change threaten coastal cities like Mumbai and New York.
   3. Deforestation in the Amazon is accelerating global climate change.


In [10]:
# Step d: MMR may choose from all 14 facts
mmr_all = facts.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,                                             # return 3 Documents ...
        "fetch_k": 14,                                      # ... chosen from all 14 facts
        "lambda_mult": 0.5,
    },
)
print("MMR, k=3 from all 14 facts (fetch_k=14):")
show(mmr_all.invoke(question))

MMR, k=3 from all 14 facts (fetch_k=14):
   1. Glaciers in the Arctic are melting at an alarming rate due to rising temperatures.
   2. LangChain supports Chroma, FAISS, Pinecone, and more.
   3. Rome is the capital of Italy


- **Similarity** returns the two glacier facts, which say the same thing twice.
- **MMR with `fetch_k=5`** keeps one of them and uses the free place for another climate fact (the Amazon).
- **MMR with `fetch_k=14`** may choose from *all* facts, so it picks unrelated ones simply because they are "different".

`fetch_k` decides where MMR may look for variety: keep it small enough that all candidates are still relevant (often 2 to 5 times `k` in a large collection), and `lambda_mult` around 0.5 to 0.8.

# <span style='background :lightgreen' >4. Retrievers without a Vector Store</span>

## <span style='color:#7e22ce'>Example 4: <code>WikipediaRetriever</code></span>

A retriever does not need a vector store of our own. `WikipediaRetriever` sends the question to Wikipedia's search and returns the best articles as Documents. No embeddings, no loading, no splitting: useful for general knowledge, and always up to date.

In [11]:
# Step a: a retriever that searches Wikipedia
import wikipedia                                            # the library that WikipediaRetriever uses
from langchain_community.retrievers import WikipediaRetriever

# Wikipedia blocks the library's default User-Agent (HTTP 403/429), so we say who we are
wikipedia.set_user_agent("GenAI-Course-Lec37/1.0 (University of the Punjab; educational use)")

wiki = WikipediaRetriever(
    top_k_results=2,                                        # the 2 best articles
    lang="en",                                              # English Wikipedia
    doc_content_chars_max=2000,                             # keep the first 2000 characters of each
)

In [12]:
# Step b: one question -> the best articles, as Documents
wiki_docs = wiki.invoke("Durand Line agreement")
for doc in wiki_docs:
    print(doc.metadata["title"], "|", doc.metadata["source"])   # the article title and its URL
    start = doc.page_content[:200].replace("\n", " ")
    print("   ", start, "...\n")

Durand Line | https://en.wikipedia.org/wiki/Durand_Line
    The Durand Line, also known as the Afghanistan–Pakistan International border, is a 2,640-kilometre (1,640 mi) international border between Afghanistan and Pakistan. The western end runs to the border  ...

Parachinar | https://en.wikipedia.org/wiki/Parachinar
    Parachinar (Pashto: پاړه چنار; Urdu: پارا چنار) is a city and the headquarters of the Kurram District in the province of Khyber Pakhtunkhwa, Pakistan. Parachinar, situated on the west of Peshawar, jut ...



> **Note:** Wikipedia allows only a few requests per minute from one program; run this cell once and reuse `wiki_docs`. Other web retrievers work the same way, for example search APIs such as Tavily. (`ArxivRetriever` for research papers currently breaks with the newest `arxiv` package, version 4.)

# <span style='background :lightgreen' >5. Better Questions, Better Ranking</span>

Back to the question that failed in Lec-36, Example 6. The first cell below rebuilds the `paper` store (Examples 5 and 6 rebuild it again, so each can run on its own). Plain similarity search returns three chunks of the **reference list**, not page 1 with the author names:

In [13]:
# Rebuild the paper store of Lec-36: load attention.pdf, split it to 200 MiniLM tokens, store the chunks
import transformers                                         # the library under sentence-transformers
transformers.logging.set_verbosity_error()                  # hide library warnings
transformers.logging.disable_progress_bar()                 # hide the "Loading weights" progress bar
from transformers import AutoTokenizer                      # to size chunks in MiniLM tokens (Lec-35)
from langchain_huggingface import HuggingFaceEmbeddings     # the free local embedding model
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=MODEL)
tokenizer = AutoTokenizer.from_pretrained(MODEL)

loader = PyPDFLoader("../data/attention.pdf")
pages = loader.load()                                       # 15 Documents, one per page
splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,                                              # measure length in MiniLM tokens
    chunk_size=200,                                         # at most 200 tokens per chunk
    chunk_overlap=20,                                       # about 10% overlap
)
chunks = splitter.split_documents(pages)
paper = InMemoryVectorStore.from_documents(chunks, embeddings)   # embed and store every chunk
print(len(chunks), "chunks of the paper are stored")

def show(results, width=95):
    """Print each Document: its page number, then the start of its text on one line."""
    number = 0
    for doc in results:
        number += 1
        page = doc.metadata["page_label"]                   # the printed page number
        text = doc.page_content[:width].replace("\n", " ")  # the start of the text, on one line
        print(f"   {number}. page {page:>2} | {text}")

60 chunks of the paper are stored


In [14]:
# Plain similarity search for the question that failed in Lec-36
QUESTION = "Who are the authors of the paper?"
dense = paper.as_retriever(search_kwargs={"k": 3})          # the 3 most similar chunks
show(dense.invoke(QUESTION))

   1. page 11 | Information Processing Systems, (NIPS), 2016. [17] Łukasz Kaiser and Ilya Sutskever. Neural GPU
   2. page 12 | [25] Mitchell P Marcus, Mary Ann Marcinkiewicz, and Beatrice Santorini. Building a large annota
   3. page 12 | Macherey, Maxim Krikun, Yuan Cao, Qin Gao, Klaus Macherey, et al. Google’s neural machine trans


## <span style='color:#7e22ce'>Example 5: <code>MultiQueryRetriever</code>: an LLM asks the question in several ways</span>

A question can be written in many ways, and each wording lands on a slightly different point in the vector space. `MultiQueryRetriever` asks an LLM for **three versions** of the question, runs the base retriever with each one, and returns the **union** of all results (without duplicates).

The `llm_chain` is a small chain from Lec-33: prompt, model, and a function that returns a `list[str]`. We use `with_structured_output()` so that the model returns a real list. (`MultiQueryRetriever.from_llm()` builds a chain that expects one question per line; our model sometimes writes the line breaks as the two characters `\n`, and then the retriever searches with one long, garbled question.)

In [15]:
# Step a: rebuild the paper store, and the plain retriever
import transformers                                         # the library under sentence-transformers
transformers.logging.set_verbosity_error()                  # hide library warnings
transformers.logging.disable_progress_bar()                 # hide the "Loading weights" progress bar
from transformers import AutoTokenizer                      # to size chunks in MiniLM tokens (Lec-35)
from langchain_huggingface import HuggingFaceEmbeddings     # the free local embedding model
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=MODEL)
tokenizer = AutoTokenizer.from_pretrained(MODEL)

loader = PyPDFLoader("../data/attention.pdf")
pages = loader.load()                                       # 15 Documents, one per page
splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,                                              # measure length in MiniLM tokens
    chunk_size=200,                                         # at most 200 tokens per chunk
    chunk_overlap=20,                                       # about 10% overlap
)
chunks = splitter.split_documents(pages)
paper = InMemoryVectorStore.from_documents(chunks, embeddings)   # embed and store every chunk
print(len(chunks), "chunks of the paper are stored")

def show(results, width=95):
    """Print each Document: its page number, then the start of its text on one line."""
    number = 0
    for doc in results:
        number += 1
        page = doc.metadata["page_label"]                   # the printed page number
        text = doc.page_content[:width].replace("\n", " ")  # the start of the text, on one line
        print(f"   {number}. page {page:>2} | {text}")

QUESTION = "Who are the authors of the paper?"
dense = paper.as_retriever(search_kwargs={"k": 3})          # plain similarity search on the paper

60 chunks of the paper are stored


In [16]:
# Step b: a small chain that writes three versions of the question (Lec-33)
from dotenv import load_dotenv                              # reads the API keys from a file
from pydantic import BaseModel, Field                       # to describe the LLM's answer (Lec-30, Example 9)
from langchain.chat_models import init_chat_model           # creates a chat model from "provider:model"
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda         # turns a Python function into a chain step

load_dotenv("../keys/.env", override=True)                  # put the keys from the file into os.environ
model = init_chat_model(
    "groq:openai/gpt-oss-120b",                             # free, on Groq
    # "openai:gpt-4o-mini",                                 # paid, on OpenAI: uncomment this line and comment out the line above
    max_retries=5,                                          # wait and retry when the free rate limit is hit
)

class Questions(BaseModel):
    questions: list[str] = Field(description="three different ways to ask the user's question")

rewrite_prompt = ChatPromptTemplate.from_template(
    "Write three different versions of this question, to search a document with a vector database. "
    "Question: {question}"
)
questions_model = model.with_structured_output(
    Questions,                                              # the answer is a Questions object
    method="json_schema",                                   # ask Groq for JSON that matches Questions
)

def get_questions(result):
    """Questions object -> a plain list of str, which MultiQueryRetriever expects."""
    return result.questions

rewrite_chain = rewrite_prompt | questions_model | RunnableLambda(get_questions)   # -> list[str]

In [17]:
# Step c: MultiQueryRetriever: search with every version, and return the union
import sys                                                  # to print the log to the cell output
import logging                                              # to see the questions the LLM writes
from langchain_classic.retrievers import MultiQueryRetriever

log = logging.getLogger("langchain_classic.retrievers.multi_query")   # the retriever logs the questions it uses
log.setLevel(logging.INFO)
if not log.handlers:                                        # add the handler only once
    log.addHandler(logging.StreamHandler(sys.stdout))

multi = MultiQueryRetriever(
    retriever=dense,                                        # the plain retriever of Step a
    llm_chain=rewrite_chain,                                # writes the versions of the question
)
results = multi.invoke(QUESTION)
print(f"\n{len(results)} different chunks:")
show(results)

Generated queries: ['Who wrote the paper?', 'Can you list the authors of the paper?', 'Who are the contributors of this article?']

5 different chunks:
   1. page 12 | [25] Mitchell P Marcus, Mary Ann Marcinkiewicz, and Beatrice Santorini. Building a large annota
   2. page 11 | Information Processing Systems, (NIPS), 2016. [17] Łukasz Kaiser and Ilya Sutskever. Neural GPU
   3. page  1 | Provided proper attribution is provided, Google hereby grants permission to reproduce the table
   4. page 10 | learning to align and translate. CoRR, abs/1409.0473, 2014. [3] Denny Britz, Anna Goldie, Minh-
   5. page 12 | Macherey, Maxim Krikun, Yuan Cao, Qin Gao, Klaus Macherey, et al. Google’s neural machine trans


In our run, the new wordings brought **page 1** into the results (the LLM writes different questions each time, so your list may differ). Multi-query search finds more (better **recall**), but it costs one LLM call per question and returns more Documents, which may not be in the best order. The next example fixes the order.

## <span style='color:#7e22ce'>Example 6: Reranking: fetch many, keep the best</span>

Our embedding model is a **bi-encoder**: it turns the question and each chunk into vectors **separately**, then compares the vectors. That is fast enough for millions of chunks, but the model never sees the question and the chunk together.

A **cross-encoder** reads the question **and** one chunk **together** and outputs one relevance score. It is much more accurate, but too slow to run on every chunk. So production systems use two stages:
1. **retrieve broadly**: the 20 nearest chunks with the fast bi-encoder;
2. **rerank narrowly**: score only those 20 with the cross-encoder, keep the best 3.

`ContextualCompressionRetriever` wraps a retriever and passes its results through a **compressor**; here the compressor is `CrossEncoderReranker`, with the free model `cross-encoder/ms-marco-MiniLM-L6-v2`, trained on Bing search questions.

In [18]:
# Step a: rebuild the paper store, and the plain retriever
import transformers                                         # the library under sentence-transformers
transformers.logging.set_verbosity_error()                  # hide library warnings
transformers.logging.disable_progress_bar()                 # hide the "Loading weights" progress bar
from transformers import AutoTokenizer                      # to size chunks in MiniLM tokens (Lec-35)
from langchain_huggingface import HuggingFaceEmbeddings     # the free local embedding model
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=MODEL)
tokenizer = AutoTokenizer.from_pretrained(MODEL)

loader = PyPDFLoader("../data/attention.pdf")
pages = loader.load()                                       # 15 Documents, one per page
splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,                                              # measure length in MiniLM tokens
    chunk_size=200,                                         # at most 200 tokens per chunk
    chunk_overlap=20,                                       # about 10% overlap
)
chunks = splitter.split_documents(pages)
paper = InMemoryVectorStore.from_documents(chunks, embeddings)   # embed and store every chunk
print(len(chunks), "chunks of the paper are stored")

def show(results, width=95):
    """Print each Document: its page number, then the start of its text on one line."""
    number = 0
    for doc in results:
        number += 1
        page = doc.metadata["page_label"]                   # the printed page number
        text = doc.page_content[:width].replace("\n", " ")  # the start of the text, on one line
        print(f"   {number}. page {page:>2} | {text}")

QUESTION = "Who are the authors of the paper?"
dense = paper.as_retriever(search_kwargs={"k": 3})          # plain similarity search on the paper

60 chunks of the paper are stored


In [19]:
# Step b: fetch 20 chunks with the fast bi-encoder, then rerank them with a cross-encoder and keep 3
from langchain_community.cross_encoders import HuggingFaceCrossEncoder   # runs a cross-encoder locally
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L6-v2")   # about 90 MB, once

broad = paper.as_retriever(search_kwargs={"k": 20})         # stage 1: 20 candidates, fast
reranker = CrossEncoderReranker(model=cross_encoder, top_n=3)   # stage 2: re-score them, keep 3
rerank = ContextualCompressionRetriever(
    base_retriever=broad,                                   # where the candidates come from
    base_compressor=reranker,                               # what to do with them
)
show(rerank.invoke(QUESTION))

   1. page  1 | Provided proper attribution is provided, Google hereby grants permission to reproduce the table
   2. page 12 | [25] Mitchell P Marcus, Mary Ann Marcinkiewicz, and Beatrice Santorini. Building a large annota
   3. page 12 | [29] Slav Petrov, Leon Barrett, Romain Thibaux, and Dan Klein. Learning accurate, compact, and 


In [20]:
# Step c: the cross-encoder's scores for the 3 chunks plain search returned, and for the reranker's winner
candidates = dense.invoke(QUESTION)                         # the 3 chunks of plain similarity search
reranked = rerank.invoke(QUESTION)
candidates.append(reranked[0])                              # plus the reranker's first choice

pairs = []                                                  # the cross-encoder reads (question, chunk) pairs
for doc in candidates:
    pairs.append((QUESTION, doc.page_content))
scores = cross_encoder.score(pairs)                         # one score per pair

for i in range(len(candidates)):
    doc = candidates[i]
    text = doc.page_content[:70].replace("\n", " ")
    print(f"   {scores[i]:6.2f}  page {doc.metadata['page_label']:>2} | {text}")

    -9.38  page 11 | Information Processing Systems, (NIPS), 2016. [17] Łukasz Kaiser and I
    -7.35  page 12 | [25] Mitchell P Marcus, Mary Ann Marcinkiewicz, and Beatrice Santorini
    -7.78  page 12 | Macherey, Maxim Krikun, Yuan Cao, Qin Gao, Klaus Macherey, et al. Goog
    -7.12  page  1 | Provided proper attribution is provided, Google hereby grants permissi


The cross-encoder gives **page 1 the highest score**, so it moves from outside the top 3 to first place. The reference-list chunks score lower. A cross-encoder's score is not limited to 0..1 (here they are all negative): only the order matters.

<div style="text-align:center;">
    <img src="../images/langchain-new-29.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

> **Note:** a reranker can only reorder what stage 1 found. If the right chunk is not among the 20 candidates, no reranker can bring it back, so stage 1 should be broad. Cloud rerankers (Cohere, Jina, Voyage) work the same way.

# <span style='background :lightgreen' >6. Keyword Search and Hybrid Search</span>

Every retriever so far compares **meanings**: the question and the chunks become vectors, and the closest vectors win. That fails when the answer depends on an **exact word**: a course code such as `CS-565`, an error code such as `E-1042`, a product ID or a name. To an embedding model, "CS-565" and "CS-530" are almost the same text.

**BM25** ("Best Matching 25", Robertson and others, 1994) is the classic ranking algorithm of search engines such as Lucene and Elasticsearch. It compares **words**, not meanings:
- it needs **no embedding model and no training**: only the text and a little counting;
- it is a **sparse** (lexical) method: each document is described by the words it contains, kept in an inverted index (word → the documents that contain it);
- it is fast, cheap and easy to explain: you can say exactly which words matched.

## a. The three ideas behind BM25
BM25 gives every document a score for the question by adding up one part for each **question word** the document contains. Three ideas decide how big each part is.

**1. Rare words count more (IDF, inverse document frequency).** A word found in almost every document says nothing about which document is meant; a word found in only one says a lot. With N = 1,000 documents:

| Word | Documents that contain it | IDF |
|---|---|---|
| `E-1042` | 1 | 6.50 |
| `payment` | 300 | 1.20 |
| `the` | 990 | 0.01 |

**2. Repeating a word helps, but less and less (TF saturation, set by `k₁`).** TF (term frequency) is how often the word appears in the document. The first occurrence counts most and the tenth adds little, so a document cannot win just by repeating a word. For a document of average length:

| Occurrences | `k₁ = 0` | `k₁ = 1.2` (common default) | `k₁ = 5` |
|---|---|---|---|
| 1 | 1.00 | 1.00 | 1.00 |
| 2 | 1.00 | 1.38 | 1.71 |
| 5 | 1.00 | 1.77 | 3.00 |
| 10 | 1.00 | 1.96 | 4.00 |

With `k₁ = 0` only presence matters; with `k₁` between 1.2 and 2.0 (the usual range) repeats help with diminishing returns; a large `k₁` makes counts matter almost linearly. The ceiling is `k₁ + 1`.

**3. Long documents are not favoured just for being long (length normalization, set by `b`).** A long document contains more words, so it has more chances to match. BM25 lowers the TF part of documents longer than average and raises it for shorter ones. One occurrence, `k₁ = 1.2`, average length 100 words:

| Document length | `b = 0` (ignore length) | `b = 0.75` (common default) | `b = 1` (full normalization) |
|---|---|---|---|
| 50 words (half the average) | 1.00 | 1.26 | 1.38 |
| 100 words (the average) | 1.00 | 1.00 | 1.00 |
| 200 words (double the average) | 1.00 | 0.71 | 0.65 |

> If you know **TF-IDF**: BM25 keeps its idea (importance × frequency) and adds two things, the saturation of TF and the length normalization.

## b. The formula
Read it as: for every question word, **IDF × TF part**, then add up. Question words that are not in the document add nothing.

```
score(D, Q)  =  Σ  IDF(qᵢ) × TF-part(qᵢ, D)
              qᵢ∈Q

IDF(qᵢ)  =  log(1 + (N − n(qᵢ) + 0.5) / (n(qᵢ) + 0.5))

                   f(qᵢ, D) · (k₁ + 1)
TF-part(qᵢ, D)  =  ───────────────────
                     f(qᵢ, D) + K(D)

K(D)  =  k₁ · (1 − b + b · |D| / avgdl)
```

| Symbol | Meaning |
|---|---|
| `Q`, `qᵢ` | the question, and one word of it |
| `D` | one document |
| `N` | the number of documents in the collection |
| `n(qᵢ)` | how many documents contain `qᵢ` |
| `f(qᵢ, D)` | how many times `qᵢ` appears in `D` |
| `\|D\|`, `avgdl` | the length of `D` in words, and the average length of all documents |
| `k₁` | how quickly repeats stop adding score (idea 2) |
| `b` | how strongly length is normalized, from 0 to 1 (idea 3) |

**Common defaults:** `k₁ = 1.2` and `b = 0.75` (Elasticsearch). Raise `k₁` if repeated mentions should count for more; lower `b` if long documents really are more relevant, such as full manuals.

## c. A worked example by hand
N = 1,000 documents, `avgdl` = 100 words, `k₁ = 1.2`, `b = 0.75`. Document `D` has **50 words**; it contains `E-1042` once and `payment` twice. The question is `E-1042 payment`.

`D` is shorter than average, so `K(D) = 1.2 · (1 − 0.75 + 0.75 · 50/100) = 0.75`. Then:

| Question word | IDF | f | TF part | IDF × TF part |
|---|---|---|---|---|
| `E-1042` (in 1 document) | 6.50 | 1 | 1 · 2.2 / (1 + 0.75) = 1.257 | 8.2 |
| `payment` (in 300 documents) | 1.20 | 2 | 2 · 2.2 / (2 + 0.75) = 1.600 | 1.9 |
| **Score** | | | | **≈ 10.1** |

`payment` appears twice, but the rare code still adds more than four times as much: exactly what we want for IDs and error codes.

## <span style='color:#7e22ce'>Example 7: BM25 by hand, then with <code>BM25Retriever</code></span>
### Step 1: the formula in Python
Two small functions are all of BM25. Change `k1` or `b` in Step 1c and run it again to see the table of idea 2 change.

In [21]:
# Step 1a: the two parts of BM25 as Python functions
import math                                                 # for log

def bm25_idf(N, n):
    """How rare a word is: N documents in all, n of them contain the word."""
    return math.log(1 + (N - n + 0.5) / (n + 0.5))

def bm25_tf(f, doc_len, avgdl, k1=1.2, b=0.75):
    """The TF part: f occurrences in a document of doc_len words (average length avgdl)."""
    K = k1 * (1 - b + b * doc_len / avgdl)                  # the length-adjusted constant K(D)
    return f * (k1 + 1) / (f + K)

print(f"IDF of E-1042: {bm25_idf(1000, 1):.2f}")            # in 1 of 1,000 documents
print(f"IDF of payment: {bm25_idf(1000, 300):.2f}")         # in 300
print(f"IDF of the: {bm25_idf(1000, 990):.2f}")             # in 990

IDF of E-1042: 6.50
IDF of payment: 1.20
IDF of the: 0.01


In [22]:
# Step 1b: the worked example: a 50-word document with "E-1042" once and "payment" twice
tf_code = bm25_tf(1, doc_len=50, avgdl=100)                 # E-1042: once
tf_payment = bm25_tf(2, doc_len=50, avgdl=100)              # payment: twice
print(f"TF parts: {tf_code:.3f} and {tf_payment:.3f}")

part_code = bm25_idf(1000, 1) * tf_code                     # IDF × TF part
part_payment = bm25_idf(1000, 300) * tf_payment
print(f"E-1042: {part_code:.1f}   payment: {part_payment:.1f}   score: {part_code + part_payment:.1f}")

TF parts: 1.257 and 1.600
E-1042: 8.2   payment: 1.9   score: 10.1


In [23]:
# Step 1c: the table of idea 2: repeats help less and less (a document of average length)
for f in [1, 2, 5, 10]:
    tf_k0 = bm25_tf(f, doc_len=100, avgdl=100, k1=0)        # only presence counts
    tf_k12 = bm25_tf(f, doc_len=100, avgdl=100, k1=1.2)     # the common default
    tf_k5 = bm25_tf(f, doc_len=100, avgdl=100, k1=5)        # counts matter more
    print(f"f = {f:2}:   k1=0 -> {tf_k0:.2f}   k1=1.2 -> {tf_k12:.2f}   k1=5 -> {tf_k5:.2f}")

f =  1:   k1=0 -> 1.00   k1=1.2 -> 1.00   k1=5 -> 1.00
f =  2:   k1=0 -> 1.00   k1=1.2 -> 1.38   k1=5 -> 1.71
f =  5:   k1=0 -> 1.00   k1=1.2 -> 1.77   k1=5 -> 3.00
f = 10:   k1=0 -> 1.00   k1=1.2 -> 1.96   k1=5 -> 4.00


### Step 2: the same ranking with `BM25Retriever`
Three short documents and the question `ERROR E-1042 payment failure`. We expect:
1. *"E-1042 payment failure troubleshooting guide"*: it has the rare code and two more question words;
2. *"Payment error codes and solutions"*: it has `error` and `payment`, but not the code;
3. *"General payment processing information"*: only the common word `payment`.

`BM25Retriever` (in `langchain-community`, using the `rank-bm25` library) builds the index from Documents. It splits the text into words with a `preprocess_func`. The default only splits at spaces and keeps capital letters, so "Payment" and "payment" would be different words; we give it a simple tokenizer: lower case, letters and digits only.

In [24]:
# Step 2a: three short documents, and a tokenizer for BM25
import re                                                   # regular expressions
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever   # needs the rank-bm25 package

docs = [
    Document("E-1042 payment failure troubleshooting guide"),
    Document("Payment error codes and solutions"),
    Document("General payment processing information"),
]

def words(text):
    """Lower-case words made of letters and digits: 'ERROR E-1042' -> ['error', 'e', '1042']."""
    return re.findall(r"\w+", text.lower())

question = "ERROR E-1042 payment failure"
print(words(question))

['error', 'e', '1042', 'payment', 'failure']


In [25]:
# Step 2b: build the BM25 index, and rank the documents
bm25 = BM25Retriever.from_documents(
    docs,                                                   # the documents to index
    k=3,                                                    # return the 3 best
    preprocess_func=words,                                  # how to split text into words
)
results = bm25.invoke(question)

number = 0
for doc in results:
    number += 1
    print(f"#{number}: {doc.page_content}")

#1: E-1042 payment failure troubleshooting guide
#2: Payment error codes and solutions
#3: General payment processing information


In [26]:
# Step 2c: look inside the index: the IDF of each question word, and each document's score
index = bm25.vectorizer                                     # the BM25Okapi object of the rank-bm25 library
print("k1 =", index.k1, "  b =", index.b)                   # the library's defaults

for word in words(question):
    print(f"   IDF of {word:8} {index.idf[word]:.3f}")

scores = index.get_scores(words(question))                  # one score per document, in the order of docs
for i in range(len(docs)):
    print(f"   score {scores[i]:.2f} | {docs[i].page_content}")

k1 = 1.5   b = 0.75
   IDF of error    0.511
   IDF of e        0.511
   IDF of 1042     0.511
   IDF of payment  0.080
   IDF of failure  0.511
   score 1.48 | E-1042 payment failure troubleshooting guide
   score 0.59 | Payment error codes and solutions
   score 0.09 | General payment processing information


**What happened:** the ranking is the one we expected. `e`, `1042` and `failure` appear only in the first document, so it wins; `error` lifts the second above the third.

**The library differs a little from the formula in b.** `BM25Retriever` uses `BM25Okapi` from `rank-bm25`:
- its IDF has no `1 +`: `log((N − n + 0.5) / (n + 0.5))`. That is **negative** for a word found in more than half of the documents, so such a word gets a small **floor** instead: 0.25 × the average IDF. Here `payment`, found in all three documents, gets only that floor.
- its defaults are `k1 = 1.5` and `b = 0.75`. Change them with `BM25Retriever.from_documents(docs, bm25_params={"k1": 1.2, "b": 0.75})`.

The three ideas are the same, and only the ranking matters.

## d. Where BM25 shines, and where it fails

| BM25 is strong on | BM25 is weak on |
|---|---|
| error codes and IDs: `E-1042`, `CS-565`, `RTX-5090` | **synonyms**: `car` does not match `automobile`, `images` does not match `vision` |
| names and acronyms: `OpenAI`, `S3`, `GDPR` | **word order**: "dog bites man" and "man bites dog" are the same bag of words |
| rare technical terms: `Kubernetes`, `CrashLoopBackOff` | **tokenization**: lower case, punctuation and word forms change the results |
| exact numbers: policy and invoice numbers | collections where `k₁` and `b` need **tuning** |

Dense search has the opposite strengths, which is why the two are often **combined** (Example 8, Step 3).

## <span style='color:#7e22ce'>Example 8: <code>BM25Retriever</code> on a transcript, then BM25 + vectors</span>

Now we compare BM25 with dense search on a document full of **codes and numbers**, where embeddings are weak: an M.Phil transcript (personal details removed).

### Step 1: one Document per course
`PyPDFLoader` writes every table cell on its own line (Lec-35, Example 7). A few lines of code turn each course into one short, readable Document.

In [27]:
# Step 1a: load the transcript, and look at its lines
import re                                                   # regular expressions, to find the course codes
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../data/arif_mphil_transcript_redacted.pdf")
pages = loader.load()                                       # a one-page PDF
text = pages[0].page_content

lines = []                                                  # one table cell per line, without empty lines
for line in text.splitlines():
    line = line.strip()
    if line:
        lines.append(line)
print(lines[16:25], "...")                                  # how a course looks in the extracted text

['Marks', 'Obtain', 'Grade', 'CS-541', 'Advance Software', 'Development', '3', '73', 'B'] ...


In [28]:
# Step 1b: one Document per course: code, title, credit hours, marks, grade and semester
courses = []                                                # the Documents we build
semester = None                                             # the semester we are in
i = 0                                                       # the line we are looking at
while i < len(lines):
    if re.fullmatch(r"Semester \d", lines[i]):              # "Semester 1" ... "Semester 4"
        semester = lines[i]
    if re.fullmatch(r"CS-\d{3}", lines[i]):                 # a course code starts a row, e.g. "CS-565"
        code = lines[i]
        j = i + 1                                           # the title starts on the next line ...
        while not lines[j].isdigit():                       # ... and can span several lines, up to the credit hours
            j += 1
        title = " ".join(lines[i + 1:j])
        credits = lines[j]                                  # the next three cells of the row
        marks = lines[j + 1]
        grade = lines[j + 2]
        course_text = f"{code} {title}: {credits} credit hours, {marks} marks, grade {grade} ({semester})."
        metadata = {"code": code, "semester": semester, "grade": grade}
        courses.append(Document(page_content=course_text, metadata=metadata))
        i = j + 3                                           # jump to the line after this row
        continue
    i += 1                                                  # not a course: look at the next line

print(f"{len(courses)} courses:")
for doc in courses:
    print("  ", doc.page_content)

10 courses:
   CS-541 Advance Software Development: 3 credit hours, 73 marks, grade B (Semester 1).
   CS-510 Analysis and Design of Algorithms: 3 credit hours, 88 marks, grade A+ (Semester 1).
   CS-530 Advance Web Semantics: 3 credit hours, 85 marks, grade A+ (Semester 1).
   CS-520 Advance Computer Architecture: 3 credit hours, 90 marks, grade A+ (Semester 2).
   CS-525 Advance Operating Systems: 3 credit hours, 89 marks, grade A+ (Semester 2).
   CS-500 Digital Signal Processing: 3 credit hours, 90 marks, grade A+ (Semester 2).
   CS-515 Theory of Computation: 3 credit hours, 83 marks, grade A (Semester 2).
   CS-561 Expert Systems: 3 credit hours, 86 marks, grade A+ (Semester 3).
   CS-565 Computer Vision: 3 credit hours, 83 marks, grade A (Semester 3).
   CS-691 Thesis: 6 credit hours, 85 marks, grade A+ (Semester 4).


### Step 2: BM25 and dense search on the same Documents
We split the text into words with the same `words()` tokenizer as Example 7, so "CS-565," and "cs-565" become the same words `cs` and `565`. Then we build a BM25 index and a vector store from the 10 courses.

In [29]:
# Step 2a: a BM25 index and a vector store, both built from the 10 courses
import transformers                                         # the library under sentence-transformers
transformers.logging.set_verbosity_error()                  # hide library warnings
transformers.logging.disable_progress_bar()                 # hide the "Loading weights" progress bar
from langchain_huggingface import HuggingFaceEmbeddings     # the free local embedding model
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.retrievers import BM25Retriever   # needs the rank-bm25 package

def words(text):
    """Lower-case words made of letters and digits: 'What grade in CS-565?' -> ['what', 'grade', 'in', 'cs', '565']."""
    return re.findall(r"\w+", text.lower())

bm25 = BM25Retriever.from_documents(
    courses,                                                # the 10 course Documents
    k=2,                                                    # return the 2 best
    preprocess_func=words,                                  # the same tokenizer as Example 7
)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
course_store = InMemoryVectorStore.from_documents(courses, embeddings)
course_dense = course_store.as_retriever(search_kwargs={"k": 2})   # dense search: the 2 nearest

print(words("What grade did I get in CS-565?"))

['what', 'grade', 'did', 'i', 'get', 'in', 'cs', '565']


In [30]:
# Step 2b: how much each word is worth (IDF): rare words count much more than words found in every course
idf = bm25.vectorizer.idf                                   # a dict: word -> IDF (rank-bm25's formula, Example 7)
for word in ["565", "vision", "computer", "grade", "cs"]:
    print(f"   {word:9} {idf[word]:.3f}")

   565       1.846
   vision    1.846
   computer  1.224
   grade     0.248
   cs        0.248


"565" appears in one course only, so it is worth a lot: with N = 10 courses, rank-bm25 gives it log((10 − 1 + 0.5) / (1 + 0.5)) = 1.846, and "computer" (in 2 courses) gets log(8.5 / 2.5) = 1.224. "grade" and "cs" appear in every course, so their IDF would be negative, and they get the small floor of Example 7 instead. Now three questions for both retrievers:

In [31]:
# Step 2c: the best course for three questions, from dense search and from BM25
def compare(question):
    """Print the best course found by dense (embedding) search and by BM25."""
    dense_best = course_dense.invoke(question)[0]           # the first Document of each list
    bm25_best = bm25.invoke(question)[0]
    print("Q:", question)
    print("   dense:", dense_best.page_content[:48])
    print("   BM25 :", bm25_best.page_content[:48])
    print()

compare("What grade did I get in CS-565?")                  # an exact course code
compare("CS-510 marks")                                     # an exact course code
compare("Which course was about images and cameras?")       # a meaning, with no matching words

Q: What grade did I get in CS-565?
   dense: CS-530 Advance Web Semantics: 3 credit hours, 85
   BM25 : CS-565 Computer Vision: 3 credit hours, 83 marks

Q: CS-510 marks
   dense: CS-520 Advance Computer Architecture: 3 credit h
   BM25 : CS-510 Analysis and Design of Algorithms: 3 cred

Q: Which course was about images and cameras?
   dense: CS-565 Computer Vision: 3 credit hours, 83 marks
   BM25 : CS-510 Analysis and Design of Algorithms: 3 cred



- **Codes and numbers:** BM25 finds CS-565 and CS-510 at once, because "565" and "510" are rare words that match exactly. For the embedding model, "CS-565", "CS-530" and "CS-520" are almost the same text, so dense search picks the wrong course first.
- **Meaning:** "images and cameras" shares no word with "Computer Vision", so BM25 cannot find it; the embedding model knows that the two mean nearly the same thing.

### Step 3: hybrid search, the best of both
`EnsembleRetriever` runs both retrievers and merges their two ranked lists with **Reciprocal Rank Fusion (RRF)**: each Document gets `weight / (rank + 60)` from every list it appears in, and the totals decide the final order. Only ranks are used, so BM25 scores and cosine scores never have to be compared.

In [32]:
# Step 3: hybrid search: BM25 and dense search, merged with Reciprocal Rank Fusion
from langchain_classic.retrievers import EnsembleRetriever

hybrid = EnsembleRetriever(
    retrievers=[bm25, course_dense],                        # the two retrievers of Step 2
    weights=[0.5, 0.5],                                     # equal say for both lists
)

questions = [
    "What grade did I get in CS-565?",
    "CS-510 marks",
    "Which course was about images and cameras?",
]
for question in questions:
    print("Q:", question)
    results = hybrid.invoke(question)
    number = 0
    for doc in results[:2]:                                 # the top 2 after fusion
        number += 1
        print(f"   {number}. {doc.page_content[:60]}")

Q: What grade did I get in CS-565?
   1. CS-565 Computer Vision: 3 credit hours, 83 marks, grade A (S
   2. CS-530 Advance Web Semantics: 3 credit hours, 85 marks, grad
Q: CS-510 marks
   1. CS-510 Analysis and Design of Algorithms: 3 credit hours, 88
   2. CS-520 Advance Computer Architecture: 3 credit hours, 90 mar
Q: Which course was about images and cameras?
   1. CS-510 Analysis and Design of Algorithms: 3 credit hours, 88
   2. CS-565 Computer Vision: 3 credit hours, 83 marks, grade A (S


Hybrid search puts the right course in the top 2 for **all three** questions. This is why hybrid search (BM25 + vectors) followed by a reranker (Example 6) is the most common retrieval setup in production RAG systems; many vector databases (Elasticsearch, Weaviate, Qdrant, Pinecone, MongoDB Atlas) offer it built in.

<div style="text-align:center;">
    <img src="../images/langchain-new-30.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

**How BM25 differs from every other retriever in this lecture:**

| | Dense retrievers (Examples 1 to 6) | BM25 (Examples 7, 8) |
|---|---|---|
| Compares | meanings (vectors from an embedding model) | words (exact matches) |
| Needs | an embedding model, a vector store | only the text: no model, no GPU |
| Good at | synonyms, paraphrases, other languages | codes, names, numbers, rare technical terms |
| Weak at | exact identifiers ("CS-565" vs "CS-530") | words that differ but mean the same ("images" vs "vision") |
| Explaining a result | hard: 384 numbers | easy: which words matched, and how rare they are |

# <span style='background :lightgreen' >7. Summary</span>

| Ex | Retriever | Idea | Package |
|---|---|---|---|
| 1 | `as_retriever(search_type="similarity")` | the k nearest chunks | `langchain-core` |
| 2 | `as_retriever(search_type="similarity_score_threshold")` | only chunks above a relevance score | `langchain-core` |
| 3 | `as_retriever(search_type="mmr")` | relevant **and** different | `langchain-core` |
| 4 | `WikipediaRetriever` | search an outside source | `langchain-community` |
| 5 | `MultiQueryRetriever` | an LLM rewrites the question several times | `langchain-classic` |
| 6 | `ContextualCompressionRetriever` + `CrossEncoderReranker` | fetch 20, rerank, keep 3 | `langchain-classic`, `langchain-community` |
| 7 | `BM25Retriever` | keyword search: rare words count more, repeats less and less, length normalized | `langchain-community` (`rank-bm25`) |
| 8 | `BM25Retriever`, `EnsembleRetriever` | keyword search on codes, and keywords + vectors | `langchain-community`, `langchain-classic` |

### Key takeaways
- A **retriever** takes a question and returns Documents; every retriever is a Runnable with `invoke()`.
- `as_retriever()` turns a vector store into a retriever: similarity, a score threshold, or MMR.
- Wrappers improve a retriever: **multi-query** finds more, a **reranker** puts the best first.
- **BM25** matches words, not meanings: rare words count more, repeats count less and less, and long documents are not favoured. It is strong on codes and names, blind to synonyms, and it needs no model.
- **Hybrid search + reranking** is the common production recipe.
- **Next lecture (Lec-38):** put a retriever, a prompt and a model in one chain, and compare it with RAG that uses no vectors at all.

### Check your understanding
1. What goes into a retriever and what comes out? Which method do you call?
2. Why can `"similarity_score_threshold"` return an empty list, and why is that useful?
3. What do `k`, `fetch_k` and `lambda_mult` control in MMR?
4. How does `MultiQueryRetriever` find more chunks than plain similarity search? What does it cost?
5. What is the difference between a bi-encoder and a cross-encoder? Why do we use both, in two stages?
6. Why can a reranker not fix a search whose candidates do not contain the right chunk?
7. What are the three ideas behind BM25, and which settings control the second and the third?
8. In the worked example of Section 6c, why does `E-1042` add more than four times as much to the score as `payment`, although `payment` appears twice?
9. Why does BM25 give "565" a much higher weight than "grade"?
10. Why did dense search confuse CS-565 with another course, and why did BM25 miss "images and cameras"?
11. How does Reciprocal Rank Fusion combine two ranked lists, and why does it not need the scores?